# Context

- Agent中的数据
    - messages: 对话数据
    - config: langchain的专有数据
    - context: 用户传递的特殊参数
        - 在工具调用过程中需要加入的额外参数
    - runtime: 执行环境数据

- 使用
    - 定义参数的格式Schema
    - 在invoke根据格式传递参数
    - 在tool、middleware通过runtime.context访问调用 

In [2]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain.tools import tool, ToolRuntime
from pydantic import BaseModel

In [3]:
@dataclass
class ContextSchema:
    user_name: str

In [4]:
# 在工具中使用
@tool
def get_weather(city: str, runtime: ToolRuntime[ContextSchema]) -> str:
    """获取城市天气。"""
    # 上下文访问
    user_name = runtime.context.user_name
    return city + F"天气很好！({user_name})"

In [5]:
# 在中间件使用
@dynamic_prompt
def personalized_prompt(request: ModelRequest) -> str:
    user_name = request.runtime.context.user_name
    return f"你是一位乐于助人的助手。请称呼用户为 {user_name}。"

In [7]:
agent = create_agent(
    model="ollama:qwen2.5:7b-instruct-q4_K_M",
    tools=[get_weather],
    middleware=[personalized_prompt],
    context_schema=ContextSchema
)

agent.invoke(
    {"messages": [{"role": "user", "content": "杭州天气怎样？"}]},
    context=ContextSchema(user_name="靓仔")
)

{'messages': [HumanMessage(content='杭州天气怎样？', additional_kwargs={}, response_metadata={}, id='32036a87-bb6f-47a1-a632-26f093c79448'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b-instruct-q4_K_M', 'created_at': '2026-08-25T03:48:20.0771661Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7549013600, 'load_duration': 6733695300, 'prompt_eval_count': 147, 'prompt_eval_duration': 227763000, 'eval_count': 20, 'eval_duration': 577977000, 'logprobs': None, 'model_name': 'qwen2.5:7b-instruct-q4_K_M', 'model_provider': 'ollama'}, id='lc_run--01a03708-666f-7ea1-bef0-7e4810bc9bdc-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '杭州'}, 'id': 'd995ef9b-0354-4e69-9bcd-51f89b54508c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 147, 'output_tokens': 20, 'total_tokens': 167}),
  ToolMessage(content='杭州天气很好！(靓仔)', name='get_weather', id='9c3e1e05-2959-4dac-973e-2097fcd5b3db', tool_call_id='d995ef9b-0354-4e69-9bc